# Case Study §7 — Evaluation scorecard (before vs after SFT)

Runnable twin of [`07_eval.py`](07_eval.py). On a 135M model you measure **relative** effects, never
claim SOTA. This bundles the project's metrics into one scorecard and runs it on the baseline instruct
model vs the SFT'd instruct model:
- **domain perplexity** — held-out chemistry text (lower better)
- **completion perplexity** — held-out Q&A answers given prompts (lower better)
- **keyword recall** — fraction of gold-answer content words the generation reproduces (higher better)
- **sample generations** — eyeball a few

> TRIAL barely trains (before ≈ after, generations may be empty) — run `MODE="full"` for real movement.

In [ ]:
MODE = "trial"     # "trial" or "full"
FORCE = False
import importlib.util, pathlib, sys, json
HERE = pathlib.Path.cwd()
root = next(p for p in [HERE, HERE/'scripts', HERE.parent/'scripts'] if (p/'config.py').exists())
sys.path.insert(0, str(root))
import config; config.set_mode(MODE)
spec = importlib.util.spec_from_file_location("s7", root / "07_eval.py")
s7 = importlib.util.module_from_spec(spec); spec.loader.exec_module(s7)
print(f"mode={config.RUN_MODE}")

## The keyword-recall rubric, demonstrated
A simple, interpretable rubric: what fraction of the gold answer's content words does a generation contain?

In [ ]:
gold = 'Density functional theory reformulates the problem in terms of the electron density.'
print('full match  :', s7.keyword_recall(gold, gold))
print('partial     :', round(s7.keyword_recall(gold, 'It uses the electron density.'), 2))
print('off-topic   :', s7.keyword_recall(gold, 'The cat sat on the mat.'))

## Run the scorecard (before vs after SFT)
Self-sufficient: trains the §5 SFT adapter if missing. Idempotent (cached to `outputs/eval_<mode>.json`).

In [ ]:
m = s7.run(force=FORCE)
print(json.dumps(m['scorecards'], indent=2))

## Verify

In [ ]:
b, a = m['scorecards']['before'], m['scorecards']['after']
for k in ('domain_ppl','completion_ppl','keyword_recall'):
    assert k in b and k in a, k
assert b['domain_ppl'] > 0 and a['domain_ppl'] > 0
print('\u2713 §7 verified: scorecard computed before and after SFT (domain ppl, completion ppl, keyword recall).')
print('In FULL mode expect completion ppl down and keyword recall up after SFT.')
print('Next: \u00a78 Unsloth-vs-HF head-to-head, then Part B (GGUF \u2192 Ollama \u2192 edge \u2192 harness).')